In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
 
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
 
data = fetch_california_housing()
df = pd.DataFrame(data['data'], columns=data['feature_names'])
df.loc[:,'target'] = data['target']
df.head(5)

In [ ]:
df.info()

In [ ]:
X = df.drop('target', axis=1)
Y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, Y, random_state=42)
print(f'?????? ????????? ??????? {X_train.shape}')
print(f'?????? ???????? ??????? {X_test.shape}')

In [ ]:
pipeline = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestRegressor())])
pipeline.fit(X_train, y_train)

In [ ]:
import math

def rmse(y_hat, y):
    return math.sqrt(mean_squared_error(y_hat, y))

In [ ]:
y_pred = pipeline.predict(X_test)
print(f'???????? ?? ??????? R2: { round(r2_score(y_test, y_pred),4)}')
print(f'???????? ?? RSME: {round(rmse(y_test, y_pred),4)}')

In [ ]:
pipeline.get_params()

from sklearn.pipeline import make_pipeline

make_pipeline(StandardScaler(), RandomForestRegressor())

In [ ]:
print(pipeline[1].n_estimators)
print(pipeline['rf'].n_estimators)

In [ ]:
pipeline.set_params(rf__n_estimators=200)

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {'scaler__with_mean':[True,False],
              'rf__n_estimators':[100, 200, 500]}
grid_search = GridSearchCV(pipeline, param_grid=param_grid, verbose = True)


grid_search.fit(X_train, y_train)
print(grid_search.best_estimator_)

In [ ]:
y_pred = grid_search.best_estimator_.predict(X_test)
print(f'???????? ?? ??????? R2: { round(r2_score(y_test, y_pred),4)}')
print(f'???????? ?? RSME: {round(rmse(y_test, y_pred),4)}')

### ????? 2. ????????????? ? ??????????

In [ ]:
df_wine= pd.read_csv('../data/Red.csv')
df_wine.head()

In [ ]:
df_wine.info()

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
ct = make_column_transformer(
     (StandardScaler(), ['Price']),
    (OneHotEncoder(), ['Country', 'Region']))
print(ct)

In [ ]:
from sklearn.compose import make_column_selector
import numpy as np

ct_filtered = make_column_transformer(
       (StandardScaler(), make_column_selector(dtype_include=np.number)),
       (OneHotEncoder(), make_column_selector(dtype_include=object))
)

print(ct_filtered)

In [ ]:
ct = make_column_transformer(
     (StandardScaler(), ['Price']),
    (OneHotEncoder(), ['Country']))

pipeline = Pipeline([('ct', ct), ('rf', RandomForestRegressor())])

X = df_wine[['Country', 'Price']]
y = df_wine['Rating']
pipeline.fit(X, y)

In [ ]:
pd.DataFrame(
    pipeline['ct'].transform(X).toarray(),
    columns=['Price'] + pipeline['ct'].transformers_[1][1].get_feature_names_out().tolist()
)

In [ ]:
import joblib
joblib.dump(pipeline, 'pipeline.pkl')

In [ ]:
pipeline_loaded = joblib.load('pipeline.pkl')
print(pipeline_loaded)

## ????????

In [ ]:
df_wine_train= pd.read_csv('../data/Red_train.csv')

ct = make_column_transformer(
    (OrdinalEncoder(), ['Region']),
     (StandardScaler(), ['Price']),
    (OneHotEncoder(), ['Country']))

X_train = df_wine_train[['Country', 'Price', 'Region']]
Y_train = df_wine_train['Rating']

pipeline = Pipeline([('ct', ct), ('rf', RandomForestRegressor(random_state=42))])
pipeline.fit(X_train, Y_train)

df_wine_test = pd.read_csv('../data/Red_test.csv')
X_test = df_wine_test[['Country', 'Price', 'Region']]
Y_test = df_wine_test['Rating']

y_pred = pipeline.predict(X_test)
print(f'???????? ?? RSME: {round(rmse(Y_test, y_pred),4)}')

In [ ]:
joblib.dump(pipeline, 'pipeline_vine.pkl')

In [ ]:
pipeline.set_params(rf__n_estimators=200)

pipeline.fit(X_train, Y_train)

y_pred = pipeline.predict(X_test)
print(f'???????? ?? RSME with n_estimators=200: {round(rmse(Y_test, y_pred),4)}')

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

estimators = [
    ('rr', RidgeCV()),
    ('dt',  DecisionTreeRegressor(random_state=42))
]

# ??????? ?????? ?????? ???????
stacking = StackingRegressor(
    estimators=estimators, #??????? ?????? ? ????????
    final_estimator=RandomForestRegressor( #??????????
        n_estimators=10, #?????????? ???????? ? ????????                  
        random_state=42 #?????? ?????????? ????????? ?????
    )
)

pipeline.set_params(rf=stacking)

pipeline.fit(X_train, Y_train)

y_pred = pipeline.predict(X_test)
print(f'???????? ?? RSME with stacking: {round(rmse(Y_test, y_pred),2)}')